In [1]:
import sys
sys.path.insert(0, '..')
from src.data.loader import load_policy_results
df = load_policy_results()

print(f"NB holdout service level:         {df['holdout_sl'].mean():.3f}")
print(f"Heuristic training service level: {df['service_level'].mean():.3f}")  # proxy
print(f"% SKUs beating heuristic on Q4:   {df['beats_heuristic'].mean()*100:.1f}%")

NB holdout service level:         0.997
Heuristic training service level: 0.992
% SKUs beating heuristic on Q4:   88.0%


In [2]:
import sys; sys.path.insert(0, '.')
import numpy as np
from src.data.loader import load_weekly_demand, load_sku_metadata, load_policy_results
from src.models.baselines import compute_normal_baseline

df       = load_weekly_demand(min_weeks=40)
meta     = load_sku_metadata()
results  = load_policy_results()
price_map = dict(zip(meta['sku_id'], meta['unit_price']))

sls = []
for sku in results['sku_id'].values[:200]:   # sample 200 — fast enough
    d = df[df['sku_id'] == sku]['demand'].values
    p = float(price_map.get(sku) or 2.50)
    h = 0.20 * max(p, 0.01) / 52
    r = compute_normal_baseline(sku, d, h, 50, 5)
    sls.append(r.simulation.service_level)

print(f"Normal (s,S) mean service level: {np.mean(sls):.3f}")

Normal (s,S) mean service level: 0.996
